# 179. 扩散语言模型：Masked Diffusion 的训练与迭代去噪怎样实现？

> **面试问题：扩散 LLM 与自回归 LLM 有何区别？前向 masking、时间条件、masked loss、置信去噪与 infilling 怎样写？**

## 先给结论

Masked diffusion 通过随时间把离散 token 替换为 `[MASK]` 的前向过程训练双向去噪器，生成时从全 mask 开始多轮预测并逐步固定 token。它可并行更新多个位置并支持自然 infilling，但不是一次 forward 完成；步数、remasking、长度和置信校准共同决定质量与延迟。

## 推荐回答主线

1. 定义 mask schedule `alpha(t)`、不可被破坏的条件 token 与可复现 corruption。
2. 实现带时间 embedding 的双向 Transformer 去噪器，只在本轮被 mask 的目标位置计算 loss。
3. 从全 mask 迭代采样，按置信/预算解锁位置，保持 prompt 与已锁位置不变。
4. 评估去噪准确率、步数、并行 token、重复/长度、校准和随机性，绑定 sampler recipe。

## 教学实现边界

Tiny 模型只有一层手写 self-attention，未在大语料训练，采样结果不代表语言质量；目标是展示离散 masking、双向条件和迭代状态，而非复现 LLaDA/MDLM 规模。

## 一手资料

- [Large Language Diffusion Models / LLaDA](https://arxiv.org/abs/2502.09992)
- [Simple and Effective Masked Diffusion LMs](https://arxiv.org/abs/2406.07524)
- [D3PM](https://arxiv.org/abs/2107.03006)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。

import warnings  # 导入本单元需要的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元需要的依赖。
import torch.nn.functional as F  # 导入本单元需要的依赖。
from torch import nn  # 导入本单元需要的依赖。

# 0=PAD、1=MASK；条件 prefix 在所有扩散时刻保持不变。
torch.manual_seed(179)  # 计算并保存当前步骤的中间状态。
PAD, MASK, VOCAB, DIM = 0, 1, 23, 16  # 计算并保存当前步骤的中间状态。
clean = torch.tensor([[2, 3, 4, 5, 6, 7], [8, 9, 10, 11, 12, 0]])  # 计算并保存当前步骤的中间状态。
valid = clean.ne(PAD)  # 计算并保存当前步骤的中间状态。
condition = torch.tensor([[1, 1, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0]], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。

assert clean.shape == valid.shape == condition.shape  # 用受控断言验证关键不变量。
assert not condition[~valid].any()  # 用受控断言验证关键不变量。
assert clean.min() >= 0 and clean.max() < VOCAB  # 用受控断言验证关键不变量。


## 1. 前向过程：时间越大，非条件 token 被 mask 的概率越高

最简单 schedule 令 mask probability 为 t。训练每个样本采 t，再独立 mask 可生成位置；至少 mask 一个目标能避免空 loss。条件/padding 永不被破坏。


In [ ]:
def corrupt_tokens(tokens, valid_mask, condition_mask, times, generator):  # 定义本节可复用的核心函数。
    if times.shape != (tokens.shape[0],) or not ((0 <= times) & (times <= 1)).all():  # 按当前条件选择后续控制路径。
        raise ValueError("times 必须是 batch 维且位于 [0,1]")  # 遇到非法合同立即显式失败。
    random_values = torch.rand(tokens.shape, generator=generator, device=tokens.device)  # 计算并保存当前步骤的中间状态。
    eligible = valid_mask & ~condition_mask  # 计算并保存当前步骤的中间状态。
    masked = eligible & (random_values < times[:, None])  # 计算并保存当前步骤的中间状态。
    # 仅对 t>0 的训练样本兜底一个监督位；t=0 必须严格保持恒等转移。
    for row in range(tokens.shape[0]):  # 遍历输入元素以累积或检查结果。
        if times[row] > 0 and eligible[row].any() and not masked[row].any():  # 按当前条件选择后续控制路径。
            masked[row, torch.nonzero(eligible[row], as_tuple=False)[0, 0]] = True  # 计算并保存当前步骤的中间状态。
    corrupted = tokens.clone(); corrupted[masked] = MASK  # 计算并保存当前步骤的中间状态。
    return corrupted, masked  # 返回当前分支计算出的结果。

# 条件和 padding 保持不变，每条正时间非空样本至少有目标；零时刻严格不破坏。
times = torch.tensor([0.3, 0.8])  # 计算并保存当前步骤的中间状态。
corrupted, masked = corrupt_tokens(clean, valid, condition, times, torch.Generator().manual_seed(1))  # 计算并保存当前步骤的中间状态。
zero_corrupted, zero_masked = corrupt_tokens(clean, valid, condition, torch.zeros(2), torch.Generator().manual_seed(1))  # 计算并保存当前步骤的中间状态。
assert torch.equal(corrupted[condition], clean[condition])  # 用受控断言验证关键不变量。
assert torch.equal(corrupted[~valid], clean[~valid])  # 用受控断言验证关键不变量。
assert masked.sum(1).min().item() >= 1  # 用受控断言验证关键不变量。
assert torch.equal(zero_corrupted, clean) and not zero_masked.any()  # 用受控断言验证关键不变量。

invalid_time_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    corrupt_tokens(clean, valid, condition, torch.tensor([-0.1, 1.1]), torch.Generator().manual_seed(1))  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    invalid_time_rejected = True  # 计算并保存当前步骤的中间状态。
assert invalid_time_rejected  # 用受控断言验证关键不变量。


## 2. 时间条件双向去噪器：MASK 两侧上下文都可用

与 causal LM 不同，去噪器可在有效序列内双向 attention。时间 embedding 告诉模型当前噪声强度；padding key 被 mask。这里手写 QKV 和 self-attention，不调用现成 Transformer。


In [ ]:
def masked_softmax(logits, allowed):  # 定义本节可复用的核心函数。
    allowed = torch.broadcast_to(allowed.to(torch.bool), logits.shape) & torch.isfinite(logits)  # 计算并保存当前步骤的中间状态。
    masked_logits = logits.masked_fill(~allowed, -torch.inf)  # 计算并保存当前步骤的中间状态。
    row_max = masked_logits.amax(-1, keepdim=True)  # 计算并保存当前步骤的中间状态。
    safe_max = torch.where(torch.isfinite(row_max), row_max, torch.zeros_like(row_max))  # 计算并保存当前步骤的中间状态。
    exponential = torch.exp(masked_logits - safe_max).masked_fill(~allowed, 0.0)  # 计算并保存当前步骤的中间状态。
    return exponential / exponential.sum(-1, keepdim=True).clamp_min(torch.finfo(logits.dtype).tiny)  # 返回当前分支计算出的结果。

class TinyDiffusionLM(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab, dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.token = nn.Embedding(vocab, dim, padding_idx=PAD)  # 计算并保存当前步骤的中间状态。
        self.time = nn.Sequential(nn.Linear(1, dim), nn.SiLU(), nn.Linear(dim, dim))  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)  # 计算并保存当前步骤的中间状态。
        self.out = nn.Linear(dim, vocab, bias=False)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, token_ids, valid_mask, times):  # 定义本节可复用的核心函数。
        hidden = self.token(token_ids) + self.time(times[:, None])[:, None, :]  # 计算并保存当前步骤的中间状态。
        q, k, v = self.qkv(self.norm(hidden)).chunk(3, -1)  # 计算并保存当前步骤的中间状态。
        score = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])  # 计算并保存当前步骤的中间状态。
        allowed = valid_mask[:, :, None] & valid_mask[:, None, :]  # 计算并保存当前步骤的中间状态。
        attention = masked_softmax(score, allowed)  # 计算并保存当前步骤的中间状态。
        context = attention @ v  # 计算并保存当前步骤的中间状态。
        output_logits = self.out(hidden + context).masked_fill(~valid_mask[..., None], 0.0)  # 计算并保存当前步骤的中间状态。
        return output_logits, attention  # 返回当前分支计算出的结果。

# padding query/key 概率均为零；全空样本也返回有限零 attention/logits。
model = TinyDiffusionLM(VOCAB, DIM)  # 计算并保存当前步骤的中间状态。
logits, attention = model(corrupted, valid, times)  # 计算并保存当前步骤的中间状态。
assert logits.shape == (*clean.shape, VOCAB)  # 用受控断言验证关键不变量。
assert attention.masked_select(~valid[:, None, :]).abs().sum() == 0  # 用受控断言验证关键不变量。
assert attention.masked_select((~valid[:, :, None]).expand_as(attention)).abs().sum() == 0  # 用受控断言验证关键不变量。
assert torch.allclose(attention.sum(-1)[valid], torch.ones_like(attention.sum(-1)[valid]))  # 用受控断言验证关键不变量。

empty_valid = valid.clone(); empty_valid[0] = False  # 计算并保存当前步骤的中间状态。
empty_logits, empty_attention = model(corrupted, empty_valid, times)  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(empty_attention).all() and torch.isfinite(empty_logits).all()  # 用受控断言验证关键不变量。
assert empty_attention[0].abs().sum() == 0 and empty_logits[0].abs().sum() == 0  # 用受控断言验证关键不变量。


## 3. Masked denoising loss：只监督本轮被破坏的位置

未 mask token 已作为条件输入，若也计 loss 会让复制任务支配训练。不同 t 可用 importance weight 校正 schedule；教学版先按 mask 数归一化，并检查条件 token 梯度不直接来自目标。


In [ ]:
def masked_denoising_loss(logits, targets, masked_positions, weights=None):  # 定义本节可复用的核心函数。
    if logits.shape[:-1] != targets.shape or targets.shape != masked_positions.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("logits/targets/masked_positions 形状不兼容")  # 遇到非法合同立即显式失败。
    token_loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1), reduction="none").view_as(targets)  # 计算并保存当前步骤的中间状态。
    if weights is None:  # 按当前条件选择后续控制路径。
        weights = torch.ones_like(token_loss)  # 计算并保存当前步骤的中间状态。
    active = masked_positions.float() * weights  # 计算并保存当前步骤的中间状态。
    if active.sum() <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("本 batch 没有去噪监督位置")  # 遇到非法合同立即显式失败。
    return (token_loss * active).sum() / active.sum(), token_loss  # 返回当前分支计算出的结果。

# 对 logits 直接求导，严格证明未 mask 位置梯度为零、mask 位置有监督。
probe_logits = logits.detach().clone().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
probe_loss, _ = masked_denoising_loss(probe_logits, clean, masked)  # 计算并保存当前步骤的中间状态。
probe_loss.backward()  # 计算并保存当前步骤的中间状态。
assert probe_logits.grad[~masked].abs().sum() == 0  # 用受控断言验证关键不变量。
assert probe_logits.grad[masked].abs().sum() > 0  # 用受控断言验证关键不变量。

# 主模型反传有限；空监督 batch 明确 fail-closed。
loss, token_loss = masked_denoising_loss(logits, clean, masked)  # 计算并保存当前步骤的中间状态。
model.zero_grad(set_to_none=True); loss.backward()  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(loss)  # 用受控断言验证关键不变量。
assert model.qkv.weight.grad is not None and torch.isfinite(model.qkv.weight.grad).all()  # 用受控断言验证关键不变量。
empty_target_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    masked_denoising_loss(logits.detach(), clean, torch.zeros_like(masked))  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    empty_target_rejected = True  # 计算并保存当前步骤的中间状态。
assert empty_target_rejected  # 用受控断言验证关键不变量。


## 4. Schedule 权重：噪声时刻采样分布属于目标定义

若均匀采 t，不同 t 的 mask 数和难度不同。可按理论目标设置权重或做分层采样；不能随意把高噪声样本重复更多而不承认目标变化。下面展示 inverse-mask-prob 权重并裁剪极端值。


In [ ]:
def schedule_weight(times, minimum=0.05, maximum=20.0):  # 定义本节可复用的核心函数。
    return (1.0 / times.clamp_min(minimum)).clamp_max(maximum)  # 返回当前分支计算出的结果。

# t 越小权重越大但受上限约束；所有权重为正且有限。
sample_times = torch.tensor([0.001, 0.1, 0.5, 1.0])  # 计算并保存当前步骤的中间状态。
weights = schedule_weight(sample_times)  # 计算并保存当前步骤的中间状态。
assert weights[0] == 20.0  # 用受控断言验证关键不变量。
assert torch.all(weights[:-1] >= weights[1:])  # 用受控断言验证关键不变量。
assert torch.isfinite(weights).all() and (weights > 0).all()  # 用受控断言验证关键不变量。


## 5. 迭代去噪：非法 token 屏蔽必须进入每一步主路径

从全 MASK 开始，每轮只锁定一部分高置信位置。PAD/MASK 在 softmax 前就被置为 `-inf`，因此即使模型把最高原始 logit 给非法 token，也只能提交合法预测；条件 token 每轮恢复，padding 永远不参与。


In [ ]:
def mask_illegal_logits(logits, illegal_ids):  # 定义本节可复用的核心函数。
    if any(not 0 <= token_id < logits.shape[-1] for token_id in illegal_ids):  # 按当前条件选择后续控制路径。
        raise ValueError("illegal token id 越界")  # 遇到非法合同立即显式失败。
    result = logits.clone()  # 计算并保存当前步骤的中间状态。
    result[..., sorted(set(illegal_ids))] = -torch.inf  # 计算并保存当前步骤的中间状态。
    return result  # 返回当前分支计算出的结果。

@torch.no_grad()  # 为下方定义附加声明式配置。
def confidence_unmask_step(model, state, valid_mask, condition_mask, clean_condition, step, total_steps):  # 定义本节可复用的核心函数。
    if total_steps <= 0 or not 0 <= step < total_steps:  # 按当前条件选择后续控制路径。
        raise ValueError("step/total_steps 非法")  # 遇到非法合同立即显式失败。
    time = torch.full((state.shape[0],), 1 - step / total_steps, device=state.device)  # 计算并保存当前步骤的中间状态。
    step_logits, _ = model(state, valid_mask, time)  # 计算并保存当前步骤的中间状态。
    safe_logits = mask_illegal_logits(step_logits, [PAD, MASK])  # 计算并保存当前步骤的中间状态。
    probability = torch.softmax(safe_logits, -1)  # 计算并保存当前步骤的中间状态。
    confidence, prediction = probability.max(-1)  # 计算并保存当前步骤的中间状态。
    undecided = state.eq(MASK) & valid_mask & ~condition_mask  # 计算并保存当前步骤的中间状态。
    next_state = state.clone()  # 计算并保存当前步骤的中间状态。
    for row in range(state.shape[0]):  # 遍历输入元素以累积或检查结果。
        remaining = int(undecided[row].sum())  # 计算并保存当前步骤的中间状态。
        take = max(1, math.ceil(remaining / (total_steps - step))) if remaining else 0  # 计算并保存当前步骤的中间状态。
        if take:  # 按当前条件选择后续控制路径。
            candidate_score = confidence[row].masked_fill(~undecided[row], -torch.inf)  # 计算并保存当前步骤的中间状态。
            index = torch.topk(candidate_score, min(take, remaining)).indices  # 计算并保存当前步骤的中间状态。
            next_state[row, index] = prediction[row, index]  # 计算并保存当前步骤的中间状态。
    next_state[condition_mask] = clean_condition[condition_mask]  # 计算并保存当前步骤的中间状态。
    next_state[~valid_mask] = clean_condition[~valid_mask]  # 计算并保存当前步骤的中间状态。
    return next_state  # 返回当前分支计算出的结果。

class IllegalPreferringModel(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def forward(self, token_ids, valid_mask, times):  # 定义本节可复用的核心函数。
        # 故意令 PAD/MASK 原始 logit 最高，合法 token 2 仅排第三。
        output = torch.full((*token_ids.shape, VOCAB), -20.0)  # 计算并保存当前步骤的中间状态。
        output[..., PAD] = 50.0; output[..., MASK] = 40.0; output[..., 2] = 10.0  # 计算并保存当前步骤的中间状态。
        return output, torch.zeros(token_ids.shape[0], token_ids.shape[1], token_ids.shape[1])  # 返回当前分支计算出的结果。

# 一步主路径保证进度、条件/padding 不变，新提交位置即使面对恶意 logits 仍只含合法 token。
state = torch.where(condition, clean, torch.where(valid, torch.full_like(clean, MASK), clean))  # 计算并保存当前步骤的中间状态。
next_state = confidence_unmask_step(model.eval(), state, valid, condition, clean, 0, 3)  # 计算并保存当前步骤的中间状态。
adversarial_next = confidence_unmask_step(IllegalPreferringModel(), state, valid, condition, clean, 0, 3)  # 计算并保存当前步骤的中间状态。
newly_fixed = state.eq(MASK) & adversarial_next.ne(MASK) & valid & ~condition  # 计算并保存当前步骤的中间状态。
assert next_state.eq(MASK).sum() < state.eq(MASK).sum()  # 用受控断言验证关键不变量。
assert torch.equal(next_state[condition], clean[condition]) and torch.equal(next_state[~valid], clean[~valid])  # 用受控断言验证关键不变量。
assert newly_fixed.any() and ((adversarial_next[newly_fixed] != PAD) & (adversarial_next[newly_fixed] != MASK)).all()  # 用受控断言验证关键不变量。


## 6. 完整 reverse loop：终止、条件与长度合同一起验收

单步正确不足以证明 sampler 正确。完整循环要验证 MASK 数单调下降并最终清零，同时左右条件、padding、序列形状都保持不变。用故意偏好非法 token 的模型做回归，可确认屏蔽逻辑确实位于循环主路径。


In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def reverse_denoise(model, clean_condition, valid_mask, condition_mask, total_steps):  # 定义本节可复用的核心函数。
    if not isinstance(total_steps, int) or total_steps <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("total_steps 必须为正整数")  # 遇到非法合同立即显式失败。
    state = torch.where(condition_mask, clean_condition, torch.where(valid_mask, torch.full_like(clean_condition, MASK), clean_condition))  # 计算并保存当前步骤的中间状态。
    history = [state.clone()]  # 计算并保存当前步骤的中间状态。
    for step in range(total_steps):  # 遍历输入元素以累积或检查结果。
        state = confidence_unmask_step(model, state, valid_mask, condition_mask, clean_condition, step, total_steps)  # 计算并保存当前步骤的中间状态。
        history.append(state.clone())  # 计算并保存当前步骤的中间状态。
    return state, history  # 返回当前分支计算出的结果。

# 完整 adversarial reverse loop 仍会终止，且所有生成位置最终都是合法 token。
final_state, reverse_history = reverse_denoise(IllegalPreferringModel(), clean, valid, condition, total_steps=3)  # 计算并保存当前步骤的中间状态。
mask_counts = [int(snapshot.eq(MASK).sum()) for snapshot in reverse_history]  # 计算并保存当前步骤的中间状态。
generated = valid & ~condition  # 计算并保存当前步骤的中间状态。
assert all(left >= right for left, right in zip(mask_counts, mask_counts[1:])) and mask_counts[-1] == 0  # 用受控断言验证关键不变量。
assert torch.equal(final_state[condition], clean[condition])  # 用受控断言验证关键不变量。
assert torch.equal(final_state[~valid], clean[~valid]) and final_state.shape == clean.shape  # 用受控断言验证关键不变量。
assert ((final_state[generated] != PAD) & (final_state[generated] != MASK)).all()  # 用受控断言验证关键不变量。
assert len(reverse_history) == 4  # 用受控断言验证关键不变量。

zero_step_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    reverse_denoise(IllegalPreferringModel(), clean, valid, condition, total_steps=0)  # 计算并保存当前步骤的中间状态。
except ValueError:  # 捕获预期异常并验证失败分支。
    zero_step_rejected = True  # 计算并保存当前步骤的中间状态。
assert zero_step_rejected  # 用受控断言验证关键不变量。


## 7. 评测：去噪准确率、并行度和步数共同决定价值

离线报告按 t 的 masked accuracy、NLL、校准与 corruption seed；生成报告 steps、每轮提交 token、最终 exact/semantic 质量、重复率和 wall-clock。并行预测多个位置不等于总延迟必然低。


In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def denoising_accuracy(model, corrupted, targets, mask, valid, times):  # 定义本节可复用的核心函数。
    safe = mask_illegal_logits(model(corrupted, valid, times)[0], [PAD, MASK])  # 计算并保存当前步骤的中间状态。
    prediction = safe.argmax(-1)  # 计算并保存当前步骤的中间状态。
    return float((prediction[mask] == targets[mask]).float().mean())  # 返回当前分支计算出的结果。

def ideal_parallelism(length, steps):  # 定义本节可复用的核心函数。
    return length / steps  # 返回当前分支计算出的结果。

# 指标位于 [0,1]，步数越少理想每步并行 token 越多，但不说明质量。
accuracy = denoising_accuracy(model.eval(), corrupted, clean, masked, valid, times)  # 计算并保存当前步骤的中间状态。
assert 0 <= accuracy <= 1  # 用受控断言验证关键不变量。
assert ideal_parallelism(16, 4) > ideal_parallelism(16, 8)  # 用受控断言验证关键不变量。
assert ideal_parallelism(16, 4) == 4  # 用受控断言验证关键不变量。


## 8. 制品门禁：forward schedule 与 reverse sampler 必须配套

mask schedule、时间参数化、loss weight、非法 token、步数、解锁/remask 策略和长度协议共同定义模型。换 sampler 可能改变输出分布，不能只保存权重。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class DiffusionArtifact:  # 定义承载本节状态与行为的数据结构。
    mask_token: int  # 执行当前语句以推进本节示例。
    schedule: str  # 执行当前语句以推进本节示例。
    time_parameterization: str  # 执行当前语句以推进本节示例。
    sampler: str  # 执行当前语句以推进本节示例。
    steps: int  # 执行当前语句以推进本节示例。
    length_policy: str  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# artifact 明确记录零时刻恒等 schedule 与集成非法 token 屏蔽的完整 reverse loop。
artifact = DiffusionArtifact(MASK, "linear-mask-zero-identity-v2", "continuous-t", "confidence-illegal-mask-loop-v2", 16, "fixed-span")  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert artifact.mask_token < VOCAB  # 用受控断言验证关键不变量。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert digest != artifact_hash(DiffusionArtifact(MASK, artifact.schedule, artifact.time_parameterization, artifact.sampler, 32, artifact.length_policy))  # 用受控断言验证关键不变量。


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
